# Modelleme — IEEE-CIS Fraud Detection

`02_feature_engineering.ipynb`'de üretilen özellikler (`ml/src/features.py`) kullanılarak model eğitimi, temporal (walk-forward) doğrulama, class-weight ile dengesizlik yönetimi, threshold optimizasyonu ve SHAP ile açıklanabilirlik bu notebook'ta ele alınır. Deneyler MLflow ile takip edilir.

In [1]:
import sys
sys.path.append("../src")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import mlflow

from features import (
    add_uid,
    add_avg_transaction_amount,
    add_amount_deviation_from_user,
    add_transaction_counts,
    add_new_device,
    add_new_location,
    add_distance_deviation_from_user,
    add_time_since_last_transaction,
    add_merchant_risk,
)

pd.set_option("display.max_columns", 50)

2026/09/19 16:08:52 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at C:\Users\Lenovo\Desktop\fraudDetection\ml\.venv\Lib\site-packages\mlflow\assistant\skills\instrumenting-with-mlflow-tracing\SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


## Veri Yükleme ve Feature Engineering

`02_feature_engineering.ipynb` ile aynı sıra — tüm leakage-safe özellikler tek seferde uygulanır.

In [2]:
df_transaction = pd.read_csv("../data/train_transaction.csv")
df_identity = pd.read_csv("../data/train_identity.csv")
df = pd.merge(df_transaction, df_identity, on="TransactionID", how="left")

df = add_uid(df)
df = add_avg_transaction_amount(df)
df = add_amount_deviation_from_user(df)
df = add_transaction_counts(df)
df = add_new_device(df)
df = add_new_location(df)
df = add_distance_deviation_from_user(df)
df = add_time_since_last_transaction(df)
df = add_merchant_risk(df)

df.shape

(590540, 448)

## MLflow Tracking Kurulumu

Deneyler proje kökünde `ml/mlruns/` altında local file-store olarak takip edilir (`.gitignore`'da zaten hariç tutuluyor). Her model/hyperparameter denemesi ayrı bir run olarak kaydedilecek; parametreler, metrikler (PR-AUC, threshold-sonrası maliyet) ve modelin kendisi (`mlflow.sklearn.log_model` / `mlflow.xgboost.log_model` vb.) loglanacak.

In [3]:
mlflow.set_tracking_uri("sqlite:///../mlflow.db")
mlflow.set_experiment("fraud-detection")

<Experiment: artifact_location='file:C:/Users/Lenovo/Desktop/fraudDetection/ml/notebooks/mlruns/1', creation_time=1789741476384, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1789741476384, lifecycle_stage='active', name='fraud-detection', tags={}, trace_location=None, workspace='default'>

## Train / Validation / Test Split — Walk-Forward

**Yaklaşım:** İki aşamalı düşünelim:

1. **Nihai test seti (holdout):** Veriyi `TransactionDT`'ye göre sırala, en sondaki (en yeni) dilimi (örn. son %15) ayır ve bir kenara koy — bu sete walk-forward CV sırasında hiç dokunulmayacak, sadece en sonda tek seferlik nihai değerlendirme için kullanılacak.
2. **Walk-forward CV (kalan ~%85 üzerinde):** Model seçimi ve threshold optimizasyonu için, art arda genişleyen pencerelerle birden fazla (train, validation) çifti üretilecek — her adımda train penceresi büyür, validation her zaman train'den kronolojik olarak sonra gelir.

**İpucu — kullanabileceğin araç:** `sklearn.model_selection.TimeSeriesSplit`. Bu sınıf tam olarak "walk-forward" mantığını uyguluyor: veriyi olduğu sırada (shuffle etmeden) `n_splits` kadar parçaya bölüyor ve her katta train penceresi bir öncekinden daha büyük oluyor, validation her zaman train'in kronolojik olarak hemen sonrasında. `.split(X)` metodu her kat için `(train_index, test_index)` çifti üretir (bunlar pozisyonel index'lerdir — DataFrame'e `.iloc` ile erişmen gerekir, `.loc` değil).

**Görev:**
- Yukarıdaki `df`'i zaten `TransactionDT`'ye göre sıralı değilse sırala (hangi fonksiyonların zaten sıraladığını `features.py`'den hatırla — ama burada nihai/tam veri seti üzerinde tekrar emin ol).
- Adım 1'i uygula: son dilimi `df_holdout_test` olarak ayır, kalanı `df_cv_pool` yap. Boyutları (`len(...)`) ve `TransactionDT` min/max aralıklarını yazdırarak sızıntı olmadığını (holdout'un gerçekten hep `df_cv_pool`'dan sonra geldiğini) doğrula.
- Adım 2'de `TimeSeriesSplit(n_splits=...)` oluştur (kaç kat mantıklı olur, veri boyutuna göre düşün — çok fazla kat, ilk katlardaki train setini gereksiz küçültür). `.split(df_cv_pool)` ile katları gez, her katta train/validation boyutlarını ve `TransactionDT` aralıklarını yazdırarak (yine sızıntı kontrolü için) doğrula.
- `TimeSeriesSplit`'in opsiyonel `gap` parametresine de bak (dokümantasyonda) — train ile validation arasına bilerek bir boşluk bırakmak ne işe yarar, projemizde gerekli mi değil mi kısaca düşün (kod yazmadan önce bana ne düşündüğünü söyleyebilirsin).

Sıradaki (boş) hücrede bunu uygula — ben review edeceğim.

In [4]:
from sklearn.model_selection import TimeSeriesSplit

# Adım 1 — Nihai holdout test seti
# TransactionDT'ye göre sırala (bazı feature fonksiyonları df'i uid/card_id'ye
# göre yeniden sıraladığı için burada tekrar zaman sırasına dönmemiz gerekiyor).
df = df.sort_values("TransactionDT").reset_index(drop=True)

holdout_frac = 0.15
split_idx = int(len(df) * (1 - holdout_frac))

df_cv_pool = df.iloc[:split_idx].reset_index(drop=True)
df_holdout_test = df.iloc[split_idx:].reset_index(drop=True)

print(
    f"df_cv_pool:      {len(df_cv_pool):>7,} satır | "
    f"TransactionDT: {df_cv_pool['TransactionDT'].min()} - {df_cv_pool['TransactionDT'].max()}"
)
print(
    f"df_holdout_test: {len(df_holdout_test):>7,} satır | "
    f"TransactionDT: {df_holdout_test['TransactionDT'].min()} - {df_holdout_test['TransactionDT'].max()}"
)

# TransactionDT saniye hassasiyetinde olduğu için aynı saniyeye denk gelen
# işlemler sınırda train/val'a bölünebilir (gerçek leakage değil, sadece
# aynı-saniye eşitliği) — bu yüzden <= kullanıyoruz ama kaç satırın bu sınıra
# denk geldiğini de görünür kılıyoruz.
boundary_ties = (df_holdout_test["TransactionDT"] == df_cv_pool["TransactionDT"].max()).sum()
print(f"Sınırda aynı saniyeye denk gelen holdout satırı: {boundary_ties}")

assert df_cv_pool["TransactionDT"].max() <= df_holdout_test["TransactionDT"].min(), (
    "Sızıntı: holdout test seti cv_pool'dan ÖNCE başlıyor!"
)

df_cv_pool:      501,959 satır | TransactionDT: 86400 - 13151840
df_holdout_test:  88,581 satır | TransactionDT: 13151880 - 15811131
Sınırda aynı saniyeye denk gelen holdout satırı: 0


In [5]:
# Adım 2 — Walk-forward CV (df_cv_pool üzerinde)
# n_splits=5: her katta train penceresi genişler, validation her zaman
# train'in kronolojik hemen sonrası. gap=0 bıraktık çünkü isFraud bu veri
# setinde geriye dönük/finalize edilmiş bir etiket — gerçek bir sistemde
# olabilecek "etiket gecikmesi" (chargeback'in günler sonra netleşmesi) bu
# veri setinde modellenmiyor, dolayısıyla train/val arasına yapay bir
# boşluk bırakmanın somut bir gerekçesi yok.
n_splits = 5
tscv = TimeSeriesSplit(n_splits=n_splits)

for fold, (train_idx, val_idx) in enumerate(tscv.split(df_cv_pool), start=1):
    df_train = df_cv_pool.iloc[train_idx]
    df_val = df_cv_pool.iloc[val_idx]

    print(
        f"Fold {fold}: train={len(df_train):>7,} "
        f"(DT {df_train['TransactionDT'].min()}-{df_train['TransactionDT'].max()}) | "
        f"val={len(df_val):>7,} "
        f"(DT {df_val['TransactionDT'].min()}-{df_val['TransactionDT'].max()})"
    )

    # Aynı gerekçe: saniye hassasiyetinde sınır eşitliğine izin ver (<=),
    # gerçek bir zaman-sırası ihlali (val'in train'den ÖNCE başlaması) olursa yakala.
    assert df_train["TransactionDT"].max() <= df_val["TransactionDT"].min(), (
        f"Fold {fold}: sızıntı — validation, train'den ÖNCE başlıyor!"
    )

Fold 1: train= 83,664 (DT 86400-1792132) | val= 83,659 (DT 1792132-3592462)


Fold 2: train=167,323 (DT 86400-3592462) | val= 83,659 (DT 3592487-5973521)


Fold 3: train=250,982 (DT 86400-5973521) | val= 83,659 (DT 5973526-8229255)


Fold 4: train=334,641 (DT 86400-8229255) | val= 83,659 (DT 8229280-10585214)


Fold 5: train=418,300 (DT 86400-10585214) | val= 83,659 (DT 10585250-13151840)


## Feature Seçimi — Sütun Kategorileri

Modelde hangi sütunların kullanılacağına karar verdik (hibrit yaklaşım): türettiğimiz özellikler + anlaşılır ham sütunlar kesin olarak dahil; ~340 anonim sütun (`V*`, `C*`, `D*`, `M*`, `id_*`, `card1/2/3/5`) ise bir tarama modeliyle önem sırasına göre elenecek, en önemli ~25-30 tanesi seçilecek.

- **`engineered_features`**: `add_*` fonksiyonlarıyla ürettiğimiz 9 özellik (`uid` hariç — o bir kimlik, feature değil).
- **`interpretable_raw`**: doğrudan anlamı bilinen ham sütunlar (işlem tutarı, ürün kodu, kart tipi, email domain, cihaz tipi, adres/mesafe).
- **`anonymous_candidates`**: geri kalan her şey — Kaggle'ın anlamını gizli tuttuğu `V`/`C`/`D`/`M`/`id_` sütunları + `card1/2/3/5` (uid'in parçası ama tek başına anlamı belirsiz kodlar).
- **`exclude_always`**: kimlikler (`TransactionID`, `uid`, `card_id`, `merchant_id`), hedef değişken (`isFraud`), ham zaman damgası (`TransactionDT` — zaten split ve türetilmiş zaman özelliklerinde kullanıldı, ham haliyle modele vermek anlamsız mutlak zaman kalıpları öğretir), ve sadece ara hesap olan yardımcı sütunlar (`global_expanding_fraud_rate`, `avg_dist1` — bunlar `merchant_risk`/`dist1_deviation_from_user`'ı hesaplamak için kullanıldı, kendileri ayrıca feature değil) ve `DeviceInfo` (zaten `new_device`'a işlendi, ham haliyle çok yüksek kardinaliteli bir metin sütunu).

In [6]:
engineered_features = [
    "avg_transaction_amount", "amount_deviation_from_user",
    "transactions_last_10min", "transactions_last_24h",
    "new_device", "new_location", "dist1_deviation_from_user",
    "time_since_last_transaction", "merchant_risk",
]

interpretable_raw = [
    "TransactionAmt", "ProductCD", "card4", "card6",
    "P_emaildomain", "R_emaildomain", "DeviceType",
    "addr1", "addr2", "dist1", "dist2",
]

exclude_always = [
    "TransactionID", "isFraud", "TransactionDT",
    "uid", "card_id", "merchant_id",
    "global_expanding_fraud_rate", "avg_dist1",
    "DeviceInfo",
]

anonymous_candidates = [
    c for c in df_cv_pool.columns
    if c not in engineered_features + interpretable_raw + exclude_always
]

print(f"engineered_features:   {len(engineered_features)}")
print(f"interpretable_raw:     {len(interpretable_raw)}")
print(f"anonymous_candidates:  {len(anonymous_candidates)}")
print(f"exclude_always:        {len(exclude_always)}")
print(f"toplam:                {len(engineered_features) + len(interpretable_raw) + len(anonymous_candidates) + len(exclude_always)} / {df_cv_pool.shape[1]}")

engineered_features:   9
interpretable_raw:     11
anonymous_candidates:  419
exclude_always:        9
toplam:                448 / 448


## Feature Set Deneyleri — Ablation + Top-K Karşılaştırması

Sabit bir "top-25/30" kararı yerine, birkaç aday feature setini gerçek walk-forward validation PR-AUC'una göre karşılaştıracağız. Üç aşama: (1) importance çıkar, (2) aday setleri tanımla, (3) hepsini 5 fold'da eğitip karşılaştır.

### Aşama 1 — Importance çıkar (Fold 5'in train'i üzerinden)

**Amaç:** ~340 anonim sütundan (`anonymous_candidates`) hangilerinin sinyal taşıdığını kaba bir LightGBM taramasıyla görmek — bu nihai model DEĞİL, sadece bir ön-eleme aracı.

**Neden Fold 5'in train'i (Fold 1 değil)?** `df_cv_pool` içindeki en büyük/temsili train dilimi (418,300 satır) — hâlâ `df_holdout_test`'e hiç dokunmuyor, ama Fold 1'in küçük (83,664 satır) ve en erken diliminden daha genellenebilir bir tarama verir.

**Neden LightGBM?** `RandomForestClassifier` eksik değer kabul etmez, önce imputation ister. LightGBM eksik değerleri native destekler (`V`/`C`/`D` sütunlarında yoğun eksiklik var, EDA'da görmüştük) — tarama için imputation kararını nihai modele erteleyebiliriz.

**Görev:**
- `list(tscv.split(df_cv_pool))` ile tüm fold'ları listeye çevir, son elemanını (`[-1]`) al → `(train_idx, val_idx)` (bu adımda `val_idx` kullanılmayacak, sadece Fold 5'in train'ini istiyoruz).
- `X_screen` = `df_cv_pool.iloc[train_idx]`'den `engineered_features + interpretable_raw + anonymous_candidates` sütunları, `y_screen` = aynı satırların `isFraud`'u.
- **Dtype hazırlığı** (bir önceki mesajda anlattığım gibi): `object` dtype'ındaki string sütunları (`ProductCD`, `card4`, `card6`, email domain'ler, `DeviceType`, `M1-M9`, bazı `id_*`) `.astype("category")`'ye çevir; nullable `"boolean"` dtype'ındaki `new_device`/`new_location`'ı `.astype("float")`'a çevir (`True/False/<NA>` → `1.0/0.0/NaN`).
- `from lightgbm import LGBMClassifier`, `class_weight="balanced"` ile `.fit(X_screen, y_screen)`.
- `model.feature_importances_` ile bir `pd.Series(..., index=X_screen.columns).sort_values(ascending=False)` oluştur, SADECE `anonymous_candidates` alt kümesine filtrele, ilk 200'ünü `anonymous_ranked` olarak sakla (kesin kesimi henüz yapma — sıradaki adımda birkaç boyut deneyeceğiz).

In [7]:
# Dtype hazırlığı — BİR KERE, df_cv_pool üzerinde (tüm fold'larda tutarlı
# kategorik kodlama için). Eğer train/val'i AYRI AYRI .astype("category")
# yapsaydık, pandas her birinde farklı kategori kümesi/kodu üretebilirdi
# (örn. train'de "visa"=0 iken val'de "visa"=1) — LightGBM kategorik
# ayrımları kod üzerinden yaptığı için bu sessizce yanlış sonuç üretirdi.
# Tek seferde, tüm df_cv_pool üzerinde yapmak bu riski ortadan kaldırıyor.
obj_cols = df_cv_pool.select_dtypes(include="object").columns
df_cv_pool[obj_cols] = df_cv_pool[obj_cols].astype("category")

bool_cols = df_cv_pool.select_dtypes(include="boolean").columns
df_cv_pool[bool_cols] = df_cv_pool[bool_cols].astype("float")

folds = list(tscv.split(df_cv_pool))
train_idx, val_idx = folds[-1]  # Fold 5

feature_universe = engineered_features + interpretable_raw + anonymous_candidates
X_screen = df_cv_pool.iloc[train_idx][feature_universe]
y_screen = df_cv_pool.iloc[train_idx]["isFraud"]

from lightgbm import LGBMClassifier

screen_model = LGBMClassifier(class_weight="balanced", random_state=42)
screen_model.fit(X_screen, y_screen)

importances = pd.Series(
    screen_model.feature_importances_, index=X_screen.columns
).sort_values(ascending=False)
anonymous_ranked = importances[importances.index.isin(anonymous_candidates)]

print(f"Fold 5 train: {len(train_idx):,} satır, {X_screen.shape[1]} sütun")
anonymous_ranked.head(20)

C:\Users\Lenovo\AppData\Local\Temp\ipykernel_5520\2291925148.py:7: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  obj_cols = df_cv_pool.select_dtypes(include="object").columns


[LightGBM] [Info] Number of positive: 14753, number of negative: 403547


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.359567 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 35409
[LightGBM] [Info] Number of data points in the train set: 418300, number of used features: 437
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


Fold 5 train: 418,300 satır, 439 sütun


id_31    174
card2    127
card1    113
C13       95
id_33     91
id_30     87
D2        74
D15       62
C1        57
C14       52
D10       48
C6        43
card5     41
C5        38
C2        37
card3     37
D1        33
D4        32
C11       28
M6        24
dtype: int32

### Aşama 2 — Aday feature setlerini tanımla

**Görev:** `feature_sets` adında bir `dict` oluştur — her key bir senaryo adı, value ilgili sütun listesi:

- `"Engineered only"` → `engineered_features` (sadece bizim 9 özellik — ham sütun yok)
- `"Raw only"` → `interpretable_raw + anonymous_candidates` (mühendislik özellikleri YOK — sadece ham sütunlar)
- `"Raw + Engineered (All)"` → `engineered_features + interpretable_raw + anonymous_candidates` (hiçbir eleme yapılmamış hali)
- `"Top-50"` → `engineered_features + interpretable_raw + anonymous_ranked[:50]`
- `"Top-100"` → aynısı, ilk 100
- `"Top-200"` → aynısı, ilk 200

(`anonymous_ranked`, bir önceki hücrede oluşturduğun sıralı `pd.Series`'in `.index`'i — bir listeye çevirmen gerekebilir, örn. `list(anonymous_ranked.index)`.)

In [8]:
anonymous_ranked_list = list(anonymous_ranked.index)

feature_sets = {
    "Engineered only": engineered_features,
    "Raw only": interpretable_raw + anonymous_candidates,
    "Raw + Engineered (All)": engineered_features + interpretable_raw + anonymous_candidates,
    "Top-50": engineered_features + interpretable_raw + anonymous_ranked_list[:50],
    "Top-100": engineered_features + interpretable_raw + anonymous_ranked_list[:100],
    "Top-200": engineered_features + interpretable_raw + anonymous_ranked_list[:200],
}

for name, cols in feature_sets.items():
    print(f"{name:<25} {len(cols)} sütun")

Engineered only           9 sütun
Raw only                  430 sütun
Raw + Engineered (All)    439 sütun
Top-50                    70 sütun
Top-100                   120 sütun
Top-200                   220 sütun


### Aşama 3 — Her varyantı 5 fold'da eğit/değerlendir, PR-AUC tablosu çıkar

**Metrik hatırlatma:** PR-AUC = `sklearn.metrics.average_precision_score(y_true, y_pred_proba)` — proje kararımızdaki birincil metrik ([[ml_decisions]]), %3.5 fraud oranındaki dengesiz veri setinde ROC-AUC'tan daha güvenilir (ROC-AUC, çok sayıdaki negatif sınıfın kolay ayırt edilmesinden şişebilir; PR-AUC azınlık sınıfa/pozitife odaklanır).

**Görev:**
- Sonuçları toplamak için boş bir liste aç: `results = []`.
- İç içe iki döngü kur: dıştaki `for set_name, cols in feature_sets.items():`, içteki `for fold, (train_idx, val_idx) in enumerate(tscv.split(df_cv_pool), start=1):`.
- Her iterasyonda: `df_train`/`df_val`'ı `.iloc[train_idx]`/`.iloc[val_idx]` ile ayır, `X_train`/`X_val`'i `cols` sütunlarıyla oluştur (Aşama 1'deki dtype hazırlığını — `object`→`category`, `boolean`→`float` — burada da uygulaman gerekiyor; üçüncü kez aynı mantığı yazacaksın, bunu bir yardımcı fonksiyona çıkarman (örn. `prepare_X(df, cols)`) hem kod tekrarını önler hem de tutarlılığı garanti eder).
- `y_train`/`y_val` = ilgili satırların `isFraud`'u.
- `LGBMClassifier(class_weight="balanced").fit(X_train, y_train)`, sonra `model.predict_proba(X_val)[:, 1]` ile pozitif sınıf olasılığını al (`predict_proba` iki sütun döner: `[:, 0]` negatif, `[:, 1]` pozitif — PR-AUC için olasılık gerekir, `predict()`'in verdiği 0/1 etiket değil).
- `average_precision_score(y_val, y_pred_proba)` ile PR-AUC hesapla, `results.append({"feature_set": set_name, "n_features": len(cols), "fold": fold, "pr_auc": pr_auc})`.
- Döngüler bitince `pd.DataFrame(results)` oluştur, `.groupby("feature_set")["pr_auc"].agg(["mean", "std"])` ile özetle (n_features'ı da ekleyip mean PR-AUC'a göre azalan sırala) ve tabloyu yorumla: hangi set en iyi ortalama PR-AUC'u veriyor, "Raw + Engineered" gerçekten "Engineered only"dan iyi mi, Top-K'lar "All" kadar iyi mi (daha az sütunla aynı performansı yakalıyorsa o daha basit/tercih edilebilir seçenek).

**Not:** Bu ~6 varyant × 5 fold = 30 LightGBM eğitimi demek; her biri muhtemelen saniyeler sürer ama toplamda birkaç dakikaya yayılabilir — sabırlı ol, arka planda bekleyebilirsin.

In [9]:
from sklearn.metrics import average_precision_score

results = []

for set_name, cols in feature_sets.items():
    for fold, (train_idx, val_idx) in enumerate(tscv.split(df_cv_pool), start=1):
        X_train = df_cv_pool.iloc[train_idx][cols]
        X_val = df_cv_pool.iloc[val_idx][cols]
        y_train = df_cv_pool.iloc[train_idx]["isFraud"]
        y_val = df_cv_pool.iloc[val_idx]["isFraud"]

        model = LGBMClassifier(class_weight="balanced", random_state=42)
        model.fit(X_train, y_train)

        y_pred_proba = model.predict_proba(X_val)[:, 1]
        pr_auc = average_precision_score(y_val, y_pred_proba)

        results.append({
            "feature_set": set_name,
            "n_features": len(cols),
            "fold": fold,
            "pr_auc": pr_auc,
        })

    print(f"{set_name} tamamlandı.")

results_df = pd.DataFrame(results)
summary = (
    results_df.groupby("feature_set")
    .agg(n_features=("n_features", "first"), mean_pr_auc=("pr_auc", "mean"), std_pr_auc=("pr_auc", "std"))
    .sort_values("mean_pr_auc", ascending=False)
)
summary

[LightGBM] [Info] Number of positive: 2233, number of negative: 81431
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002341 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1371
[LightGBM] [Info] Number of data points in the train set: 83664, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


[LightGBM] [Info] Number of positive: 4629, number of negative: 162694
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.005061 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1425
[LightGBM] [Info] Number of data points in the train set: 167323, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


[LightGBM] [Info] Number of positive: 8175, number of negative: 242807
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.010827 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1421
[LightGBM] [Info] Number of data points in the train set: 250982, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=-0.000000
[LightGBM] [Info] Start training from score -0.000000


[LightGBM] [Info] Number of positive: 11307, number of negative: 323334
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011301 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1413
[LightGBM] [Info] Number of data points in the train set: 334641, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


[LightGBM] [Info] Number of positive: 14753, number of negative: 403547
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.012945 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1406
[LightGBM] [Info] Number of data points in the train set: 418300, number of used features: 9
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


...

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.123394 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23834
[LightGBM] [Info] Number of data points in the train set: 334641, number of used features: 220
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


[LightGBM] [Info] Number of positive: 14753, number of negative: 403547


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.161631 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 23699
[LightGBM] [Info] Number of data points in the train set: 418300, number of used features: 220
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000


Top-200 tamamlandı.


,n_features,mean_pr_auc,std_pr_auc
feature_set,,,
Raw only,430,0.540252,0.025647
Raw + Engineered (All),439,0.539309,0.025283
Top-100,120,0.538608,0.029432
Top-200,220,0.538599,0.026104
Top-50,70,0.525359,0.034563
Engineered only,9,0.137660,0.029493


### Sonuç ve Karar

Tablo: `Raw only` (0.5403) ve `Raw + Engineered (All)` (0.5393) arasındaki fark std'nin (~0.025) çok altında — istatistiksel olarak anlamsız. Yani türettiğimiz 9 özellik, ham+anonim sütunların zaten taşıdığı sinyale ölçülebilir bir PR-AUC katkısı eklemiyor (muhtemelen IEEE-CIS'in anonim `V`/`C`/`D` sütunları, bizimkine benzer davranışsal/velocity agregasyonlarını zaten örtük olarak içeriyor). Bu, feature engineering'in başarısız olduğu anlamına gelmiyor — 9 özelliğimiz SHAP/rapor aşamasında çok daha yüksek **açıklanabilirlik** değeri taşıyacak (`merchant_risk=0.8` demek, `V204=1.3` demekten çok daha anlaşılır bir bankacılık gerekçesi).

`Top-100` (120 sütun, 0.5386) ile `All` (439 sütun, 0.5393) arasındaki fark da gürültü seviyesinde — çok daha az sütunla neredeyse aynı performans. **Karar: nihai feature seti `Top-100`** (9 engineered + 11 interpretable + 100 top anonim = 120 sütun) — SHAP analizi ve rapor için çok daha yönetilebilir, performans kaybı anlamsız.

In [10]:
final_features = feature_sets["Top-100"]
print(f"final_features: {len(final_features)} sütun")

final_features: 120 sütun


## Logistic Regression — Pipeline Kurulumu (Aşama A: tek fold'da sağlama)

### Bilinen sorun: `dist1_deviation_from_user`'daki `inf` değerleri

`02_feature_engineering.ipynb`'de flaglenmişti: kullanıcının geçmiş `dist1` ortalaması (`avg_dist1`) sıfırsa, yüzdesel sapma formülü (`(dist1 - avg_dist1) / avg_dist1`) sıfıra bölme nedeniyle `inf` üretiyor (834 satır). LightGBM buna takılmadı (native olarak büyük değerleri de idare ediyor) ama scikit-learn'ün `SimpleImputer`/`StandardScaler`'ı `inf` kabul etmiyor — az önceki hata da tam bu yüzden.

**Çözüm:** `inf`/`-inf`'i `NaN`'a çeviriyoruz. Bunun mantığı: `inf` değeri zaten anlamlı bir büyüklük taşımıyor (sadece "payda sıfırdı" bilgisini taşıyor), rastgele büyük bir sayıya sabitlemek yerine mevcut eksiklik-yönetimi mantığımıza (medyan ile doldurma) bırakmak daha tutarlı — tıpkı `dist1_deviation_from_user`'ın zaten `NaN` olduğu diğer satırlar gibi ele alınmış olacak.

In [11]:
n_inf_before = np.isinf(df_cv_pool["dist1_deviation_from_user"]).sum()
df_cv_pool["dist1_deviation_from_user"] = df_cv_pool["dist1_deviation_from_user"].replace(
    [np.inf, -np.inf], np.nan
)
n_inf_after = np.isinf(df_cv_pool["dist1_deviation_from_user"]).sum()
print(f"inf sayısı: {n_inf_before} -> {n_inf_after}")

inf sayısı: 732 -> 0


In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression

numeric_cols = list(df_cv_pool[final_features].select_dtypes(include=["float", "int"]).columns)
categorical_cols = list(df_cv_pool[final_features].select_dtypes(include="category").columns)

print(f"numeric_cols:     {len(numeric_cols)}")
print(f"categorical_cols: {len(categorical_cols)}")
print(f"toplam:           {len(numeric_cols) + len(categorical_cols)} / {len(final_features)}")

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("encoder", OneHotEncoder(handle_unknown="ignore")),
])

preprocessor = ColumnTransformer([
    ("num", numeric_pipeline, numeric_cols),
    ("cat", categorical_pipeline, categorical_cols),
])

lr_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
])

# Fold 5 üzerinde sağlama — tüm fold'ları henüz gezmiyoruz, sadece pipeline
# doğru çalışıyor mu diye bakıyoruz.
train_idx, val_idx = folds[-1]
X_train = df_cv_pool.iloc[train_idx][final_features]
X_val = df_cv_pool.iloc[val_idx][final_features]
y_train = df_cv_pool.iloc[train_idx]["isFraud"]
y_val = df_cv_pool.iloc[val_idx]["isFraud"]

lr_pipeline.fit(X_train, y_train)
y_pred_proba = lr_pipeline.predict_proba(X_val)[:, 1]
pr_auc = average_precision_score(y_val, y_pred_proba)

print(f"Fold 5 — Logistic Regression PR-AUC: {pr_auc:.4f}")

numeric_cols:     105
categorical_cols: 15
toplam:           120 / 120


Fold 5 — Logistic Regression PR-AUC: 0.3854


## Logistic Regression — Aşama B: 5 Fold + MLflow Loglama

In [13]:
import mlflow.sklearn


def build_lr_pipeline():
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols),
    ])

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)),
    ])


fold_scores = []

with mlflow.start_run(run_name="LogisticRegression"):
    mlflow.log_param("model", "LogisticRegression")
    mlflow.log_param("n_features", len(final_features))
    mlflow.log_param("class_weight", "balanced")

    for fold, (train_idx, val_idx) in enumerate(tscv.split(df_cv_pool), start=1):
        X_train = df_cv_pool.iloc[train_idx][final_features]
        X_val = df_cv_pool.iloc[val_idx][final_features]
        y_train = df_cv_pool.iloc[train_idx]["isFraud"]
        y_val = df_cv_pool.iloc[val_idx]["isFraud"]

        pipeline = build_lr_pipeline()
        pipeline.fit(X_train, y_train)

        y_pred_proba = pipeline.predict_proba(X_val)[:, 1]
        pr_auc = average_precision_score(y_val, y_pred_proba)

        fold_scores.append(pr_auc)
        mlflow.log_metric("pr_auc", pr_auc, step=fold)
        print(f"Fold {fold}: PR-AUC={pr_auc:.4f}")

    pr_auc_mean = float(np.mean(fold_scores))
    pr_auc_std = float(np.std(fold_scores))
    mlflow.log_metric("pr_auc_mean", pr_auc_mean)
    mlflow.log_metric("pr_auc_std", pr_auc_std)

    # MLflow 3.x, sklearn modellerini varsayılan olarak `skops` ile
    # kaydediyor (pickle yerine daha güvenli bir serileştirme formatı) —
    # skops, tanımadığı tipleri güvenlik amacıyla reddediyor. Buradaki
    # `numpy.dtype` referansı OneHotEncoder'ın iç yapısından geliyor,
    # zararsız — açıkça güvenilir olarak işaretliyoruz.
    mlflow.sklearn.log_model(pipeline, name="model", skops_trusted_types=["numpy.dtype"])

    print(f"\nLogistic Regression — Mean PR-AUC: {pr_auc_mean:.4f} (+/- {pr_auc_std:.4f})")

Fold 1: PR-AUC=0.3150


Fold 2: PR-AUC=0.4064


Fold 3: PR-AUC=0.3824


Fold 4: PR-AUC=0.4335


Fold 5: PR-AUC=0.3854



Logistic Regression — Mean PR-AUC: 0.3845 (+/- 0.0393)


## Random Forest — 5 Fold + MLflow Loglama

LR'nin pipeline yapısını yeniden kullanıyoruz, iki farkla: `StandardScaler` yok (ağaç modelleri ölçekten etkilenmez) ve kategorik sütunlar için `OneHotEncoder` yerine `OrdinalEncoder` (Random Forest, yüzlerce seyrek one-hot sütunuyla LR kadar verimli çalışmıyor; her kategoriyi tek bir tamsayı koduna çeviren `OrdinalEncoder` ağaç bölmeleri için yeterli — `handle_unknown="use_encoded_value", unknown_value=-1` ile validation'da train'de görülmemiş bir kategori çıkarsa hata vermek yerine -1 kodunu veriyor).

In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import OrdinalEncoder


def build_rf_pipeline():
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols),
    ])

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", RandomForestClassifier(
            n_estimators=100,
            class_weight="balanced",
            n_jobs=-1,
            random_state=42,
        )),
    ])


fold_scores = []

with mlflow.start_run(run_name="RandomForest"):
    mlflow.log_param("model", "RandomForest")
    mlflow.log_param("n_features", len(final_features))
    mlflow.log_param("class_weight", "balanced")
    mlflow.log_param("n_estimators", 100)

    for fold, (train_idx, val_idx) in enumerate(tscv.split(df_cv_pool), start=1):
        X_train = df_cv_pool.iloc[train_idx][final_features]
        X_val = df_cv_pool.iloc[val_idx][final_features]
        y_train = df_cv_pool.iloc[train_idx]["isFraud"]
        y_val = df_cv_pool.iloc[val_idx]["isFraud"]

        pipeline = build_rf_pipeline()
        pipeline.fit(X_train, y_train)

        y_pred_proba = pipeline.predict_proba(X_val)[:, 1]
        pr_auc = average_precision_score(y_val, y_pred_proba)

        fold_scores.append(pr_auc)
        mlflow.log_metric("pr_auc", pr_auc, step=fold)
        print(f"Fold {fold}: PR-AUC={pr_auc:.4f}")

    pr_auc_mean = float(np.mean(fold_scores))
    pr_auc_std = float(np.std(fold_scores))
    mlflow.log_metric("pr_auc_mean", pr_auc_mean)
    mlflow.log_metric("pr_auc_std", pr_auc_std)

    # Random Forest'ın ağaç yapısı (sklearn.tree._tree.Tree) da skops'un
    # varsayılan güvenilir tip listesinde yok (bounds-check yapılmadan
    # işaretçi gibi kullanılan ham node indeksleri barındırdığı için
    # skops bunu genel olarak riskli sayıyor) — kendi ürettiğimiz,
    # güvendiğimiz bir model olduğu için açıkça güvenilir işaretliyoruz.
    mlflow.sklearn.log_model(
        pipeline, name="model",
        skops_trusted_types=["numpy.dtype", "sklearn.tree._tree.Tree"],
    )

    print(f"\nRandom Forest — Mean PR-AUC: {pr_auc_mean:.4f} (+/- {pr_auc_std:.4f})")

Fold 1: PR-AUC=0.5098


Fold 2: PR-AUC=0.5841


Fold 3: PR-AUC=0.5849


Fold 4: PR-AUC=0.5805


Fold 5: PR-AUC=0.5464



Random Forest — Mean PR-AUC: 0.5612 (+/- 0.0294)


## XGBoost — 5 Fold + MLflow Loglama

LightGBM gibi eksik değerleri native destekliyor — ayrı bir imputation pipeline'ına gerek yok, `final_features` sütunları (zaten `category`/`float` dtype'ında) doğrudan kullanılabiliyor. `class_weight` yerine `scale_pos_weight` (negatif/pozitif sınıf oranı) kullanılıyor — her fold'un train'indeki dağılım farklı olduğu için bu oran her fold'da yeniden hesaplanıyor.

In [15]:
from xgboost import XGBClassifier
import mlflow.xgboost

fold_scores = []

with mlflow.start_run(run_name="XGBoost"):
    mlflow.log_param("model", "XGBoost")
    mlflow.log_param("n_features", len(final_features))

    for fold, (train_idx, val_idx) in enumerate(tscv.split(df_cv_pool), start=1):
        X_train = df_cv_pool.iloc[train_idx][final_features]
        X_val = df_cv_pool.iloc[val_idx][final_features]
        y_train = df_cv_pool.iloc[train_idx]["isFraud"]
        y_val = df_cv_pool.iloc[val_idx]["isFraud"]

        scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

        model = XGBClassifier(
            scale_pos_weight=scale_pos_weight,
            enable_categorical=True,
            random_state=42,
        )
        model.fit(X_train, y_train)

        y_pred_proba = model.predict_proba(X_val)[:, 1]
        pr_auc = average_precision_score(y_val, y_pred_proba)

        fold_scores.append(pr_auc)
        mlflow.log_metric("pr_auc", pr_auc, step=fold)
        print(f"Fold {fold}: PR-AUC={pr_auc:.4f} (scale_pos_weight={scale_pos_weight:.2f})")

    pr_auc_mean = float(np.mean(fold_scores))
    pr_auc_std = float(np.std(fold_scores))
    mlflow.log_metric("pr_auc_mean", pr_auc_mean)
    mlflow.log_metric("pr_auc_std", pr_auc_std)
    mlflow.xgboost.log_model(model, name="model")

    print(f"\nXGBoost — Mean PR-AUC: {pr_auc_mean:.4f} (+/- {pr_auc_std:.4f})")

Fold 1: PR-AUC=0.4642 (scale_pos_weight=36.47)


Fold 2: PR-AUC=0.5071 (scale_pos_weight=35.15)


Fold 3: PR-AUC=0.5283 (scale_pos_weight=29.70)


Fold 4: PR-AUC=0.5271 (scale_pos_weight=28.60)


Fold 5: PR-AUC=0.4952 (scale_pos_weight=27.35)



XGBoost — Mean PR-AUC: 0.5044 (+/- 0.0237)


## Hafif Hyperparameter Tuning — Random Forest ve LightGBM

En iyi 2 model üzerinde `RandomizedSearchCV` ile hafif bir arama (`n_iter=8`, grid search değil). `cv=tscv` vererek walk-forward fold'larımızı otomatik kullanıyoruz — elle fold döngüsü yazmaya gerek yok. `scoring="average_precision"` = PR-AUC (projenin birincil metriği).

In [16]:
from sklearn.model_selection import RandomizedSearchCV

X = df_cv_pool[final_features]
y = df_cv_pool["isFraud"]

# --- Random Forest ---
rf_param_distributions = {
    "model__n_estimators": [100, 200, 300],
    "model__max_depth": [10, 20, 30, None],
    "model__min_samples_leaf": [1, 2, 5, 10],
    "model__max_features": ["sqrt", "log2", 0.5],
}

rf_pipeline_for_search = build_rf_pipeline()
rf_pipeline_for_search.set_params(model__n_jobs=1)  # RandomizedSearchCV zaten n_jobs=-1 ile paralelleşecek

rf_search = RandomizedSearchCV(
    rf_pipeline_for_search,
    param_distributions=rf_param_distributions,
    n_iter=8,
    scoring="average_precision",
    cv=tscv,
    random_state=42,
    n_jobs=-1,
)
rf_search.fit(X, y)

print("Random Forest — en iyi parametreler:", rf_search.best_params_)
print(f"Random Forest — en iyi PR-AUC (CV ortalaması): {rf_search.best_score_:.4f}")
print(f"(Tuning öncesi default PR-AUC: 0.5612)")

with mlflow.start_run(run_name="RandomForest-Tuned"):
    mlflow.log_params(rf_search.best_params_)
    mlflow.log_metric("pr_auc_mean", rf_search.best_score_)
    mlflow.sklearn.log_model(
        rf_search.best_estimator_, name="model",
        skops_trusted_types=["numpy.dtype", "sklearn.tree._tree.Tree"],
    )

Random Forest — en iyi parametreler: {'model__n_estimators': 200, 'model__min_samples_leaf': 10, 'model__max_features': 'sqrt', 'model__max_depth': 30}
Random Forest — en iyi PR-AUC (CV ortalaması): 0.5320
(Tuning öncesi default PR-AUC: 0.5612)


In [17]:
import mlflow.lightgbm

# --- LightGBM ---
lgbm_param_distributions = {
    "num_leaves": [15, 31, 63, 127],
    "learning_rate": [0.01, 0.05, 0.1, 0.2],
    "n_estimators": [100, 200, 300],
    "min_child_samples": [10, 20, 50],
}

lgbm_search = RandomizedSearchCV(
    LGBMClassifier(class_weight="balanced", random_state=42, verbose=-1, n_jobs=1),
    param_distributions=lgbm_param_distributions,
    n_iter=8,
    scoring="average_precision",
    cv=tscv,
    random_state=42,
    n_jobs=-1,
)
lgbm_search.fit(X, y)

print("LightGBM — en iyi parametreler:", lgbm_search.best_params_)
print(f"LightGBM — en iyi PR-AUC (CV ortalaması): {lgbm_search.best_score_:.4f}")
print(f"(Tuning öncesi default PR-AUC: 0.5386)")

with mlflow.start_run(run_name="LightGBM-Tuned"):
    mlflow.log_params(lgbm_search.best_params_)
    mlflow.log_metric("pr_auc_mean", lgbm_search.best_score_)
    mlflow.lightgbm.log_model(lgbm_search.best_estimator_, name="model")

LightGBM — en iyi parametreler: {'num_leaves': 63, 'n_estimators': 300, 'min_child_samples': 10, 'learning_rate': 0.1}
LightGBM — en iyi PR-AUC (CV ortalaması): 0.5681
(Tuning öncesi default PR-AUC: 0.5386)


## Isolation Forest — Unsupervised Anomaly Detection

RF'nin preprocessing pipeline'ını (imputer + `OrdinalEncoder`) yeniden kullanıyoruz — `IsolationForest` de sayısal girdi istiyor, `NaN` kabul etmiyor. Fark: `.fit()` sırasında `y` (isFraud) HİÇ kullanılmıyor, model tamamen etiketsiz çalışıyor. Değerlendirirken (PR-AUC hesaplarken) yine gerçek etiketleri kullanıyoruz — "etiketsiz bulunan anomaliler gerçekten fraud'a denk geliyor mu?" sorusuna cevap arıyoruz.

`score_samples` yüksek değeri "normal/inlier" anlamına getiriyor (ters yönde) — bizim "risk skoru" olarak kullanmamız için negatifini alıyoruz (`-score_samples`), böylece yüksek skor = daha anormal/riskli olacak şekilde diğer modellerle tutarlı bir yön elde ediyoruz.

In [ ]:
from sklearn.ensemble import IsolationForest


def build_iso_pipeline():
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])

    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols),
    ])

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", IsolationForest(
            n_estimators=100,
            contamination=0.035,
            random_state=42,
            n_jobs=-1,
        )),
    ])


fold_scores = []

with mlflow.start_run(run_name="IsolationForest"):
    mlflow.log_param("model", "IsolationForest")
    mlflow.log_param("n_features", len(final_features))
    mlflow.log_param("contamination", 0.035)

    for fold, (train_idx, val_idx) in enumerate(tscv.split(df_cv_pool), start=1):
        X_train = df_cv_pool.iloc[train_idx][final_features]
        X_val = df_cv_pool.iloc[val_idx][final_features]
        y_val = df_cv_pool.iloc[val_idx]["isFraud"]

        pipeline = build_iso_pipeline()
        pipeline.fit(X_train)  # etiketsiz — y_train hiç kullanılmıyor

        anomaly_score = -pipeline.score_samples(X_val)  # yüksek = daha anormal/riskli
        pr_auc = average_precision_score(y_val, anomaly_score)

        fold_scores.append(pr_auc)
        mlflow.log_metric("pr_auc", pr_auc, step=fold)
        print(f"Fold {fold}: PR-AUC={pr_auc:.4f}")

    pr_auc_mean = float(np.mean(fold_scores))
    pr_auc_std = float(np.std(fold_scores))
    mlflow.log_metric("pr_auc_mean", pr_auc_mean)
    mlflow.log_metric("pr_auc_std", pr_auc_std)
    mlflow.sklearn.log_model(
        pipeline, name="model",
        skops_trusted_types=["numpy.dtype", "sklearn.tree._tree.Tree"],
    )

    print(f"\nIsolation Forest — Mean PR-AUC: {pr_auc_mean:.4f} (+/- {pr_auc_std:.4f})")

## Threshold Optimizasyonu — Maliyet Matrisi Bazlı

**Metodolojik nokta:** `lgbm_search.best_estimator_`, `RandomizedSearchCV`'nin `refit=True` davranışı yüzünden TÜM `df_cv_pool`'a zaten fit edilmiş durumda — threshold seçimi için bunu kullanırsak modelin gördüğü veriye bakmış oluruz (iyimser/yanlı). Holdout test'i de kullanamayız (tek seferlik nihai değerlendirme için saklı). Çözüm: en iyi hiperparametrelerle **Fold 5'in train'inde YENİDEN** bir model eğitip, Fold 5'in validation'ında threshold arayacağız.

**Maliyet çerçevesi — Bahnsen et al. (2013-2015), example-dependent cost-sensitive fraud detection:**
- **FN** (fraud'u kaçırmak, APPROVE + gerçek fraud): maliyet = `TransactionAmt` (kaybedilen tam tutar).
- **TP/FP** (işlem "flag" edildi — doğru ya da yanlış fark etmez): maliyet = sabit bir **idari maliyet (Ca)** — araştırma + kart sahibiyle iletişim. Doğru yakalasan bile (TP) bu maliyeti ödersin.
- **TN**: maliyet = 0.

**Bizim 3-aksiyonlu genişletmemiz (literatürün ötesi, kendi mimarimize özgü):** APPROVE/REVIEW/BLOCK — REVIEW ve BLOCK farklı maliyetlere sahip (`C_review=$5`, `C_block=$25`) çünkü otomatik bir blok, müşteri ilişkisi açısından bir incelemeden daha maliyetli varsayılıyor. Bunlar **gerçek banka verisi değil, belgelenmiş varsayımlar** — bu yüzden sonunda bir duyarlılık analizi de yapacağız.

**Eşdeğerlik notu (önemli, hassas bir ayrım):** `t_review = t_block` sadece aksiyon uzayını ikiye indirir (REVIEW ortadan kalkar), maliyetlerden bağımsız. `C_review = C_block = Ca` ise sadece maliyet yapısını Bahnsen'inkine eşitler. **İkisi birden** sağlandığında sistem tam olarak Bahnsen'in 2-sınıflı modeline eşdeğer olur — aşağıda bunu sayısal olarak doğrulayacağız.

In [ ]:
# --- Adım 1: Fold 5'in train'inde en iyi hiperparametrelerle YENİDEN eğit ---
train_idx, val_idx = folds[-1]

X_train_f5 = df_cv_pool.iloc[train_idx][final_features]
y_train_f5 = df_cv_pool.iloc[train_idx]["isFraud"]
X_val_f5 = df_cv_pool.iloc[val_idx][final_features]
y_val_f5 = df_cv_pool.iloc[val_idx]["isFraud"].values
amt_val_f5 = df_cv_pool.iloc[val_idx]["TransactionAmt"].values

threshold_model = LGBMClassifier(
    num_leaves=63, n_estimators=300, learning_rate=0.1, min_child_samples=10,
    class_weight="balanced", random_state=42, verbose=-1,
)
threshold_model.fit(X_train_f5, y_train_f5)
y_proba_val_f5 = threshold_model.predict_proba(X_val_f5)[:, 1]

print(f"Fold 5 validation: {len(y_val_f5):,} satır, {y_val_f5.sum()} fraud")
print(f"Olasılık aralığı: {y_proba_val_f5.min():.4f} - {y_proba_val_f5.max():.4f}")


# --- Adım 2: Maliyet fonksiyonu ---
def compute_total_cost(y_true, amt, proba, t_review, t_block, c_review, c_block):
    """
    APPROVE (proba < t_review): fraud ise maliyet=amt, değilse 0.
    REVIEW  (t_review <= proba < t_block): maliyet=c_review (doğru/yanlış fark etmez).
    BLOCK   (proba >= t_block): maliyet=c_block (doğru/yanlış fark etmez).
    """
    is_approve = proba < t_review
    is_review = (proba >= t_review) & (proba < t_block)
    is_block = proba >= t_block

    cost = np.zeros_like(amt, dtype=float)
    cost[is_approve] = np.where(y_true[is_approve] == 1, amt[is_approve], 0.0)
    cost[is_review] = c_review
    cost[is_block] = c_block

    n_review = is_review.sum()
    n_block = is_block.sum()
    fraud_missed = ((y_true == 1) & is_approve).sum()
    return cost.sum(), n_review, n_block, fraud_missed


THRESHOLDS = np.round(np.arange(0.01, 1.00, 0.02), 2)

print(f"\nToplam {len(THRESHOLDS)} aday eşik değeri taranacak.")

In [ ]:
# --- Adım 3: Saf Bahnsen (2-sınıflı) benchmark ---
# t_review = t_block = t VE c_review = c_block = Ca -> tam Bahnsen'e eşdeğer.
C_ADMIN_BAHNSEN = 5.0

bahnsen_rows = []
for t in THRESHOLDS:
    cost, n_rev, n_blk, n_missed = compute_total_cost(
        y_val_f5, amt_val_f5, y_proba_val_f5, t, t, C_ADMIN_BAHNSEN, C_ADMIN_BAHNSEN
    )
    bahnsen_rows.append({"threshold": t, "total_cost": cost, "fraud_missed": n_missed})

bahnsen_df = pd.DataFrame(bahnsen_rows).sort_values("total_cost").reset_index(drop=True)
best_bahnsen = bahnsen_df.iloc[0]

print("=== Bahnsen (2-sınıflı) benchmark, Ca=$5 ===")
print(f"En iyi threshold: {best_bahnsen['threshold']:.2f} -> toplam maliyet: ${best_bahnsen['total_cost']:,.2f}")
print(f"Kaçırılan fraud sayısı: {int(best_bahnsen['fraud_missed'])} / {y_val_f5.sum()}")

plt.figure(figsize=(8, 4))
plt.plot(bahnsen_df["threshold"], bahnsen_df["total_cost"])
plt.axvline(best_bahnsen["threshold"], color="red", linestyle="--", label=f"En iyi t={best_bahnsen['threshold']:.2f}")
plt.xlabel("Threshold")
plt.ylabel("Toplam Maliyet ($)")
plt.title("Bahnsen (2-sınıflı) — Threshold vs Toplam Maliyet")
plt.legend()
plt.tight_layout()
plt.savefig("../reports/threshold_bahnsen.png", dpi=100)
plt.show()


# --- Adım 4: Bizim 3-aksiyonlu genişletmemiz (Ca=5, Cblock=25) ---
C_REVIEW = 5.0
C_BLOCK = 25.0

def sweep_3tier(c_review, c_block):
    rows = []
    for t_review in THRESHOLDS:
        for t_block in THRESHOLDS:
            if t_block < t_review:
                continue
            cost, n_rev, n_blk, n_missed = compute_total_cost(
                y_val_f5, amt_val_f5, y_proba_val_f5, t_review, t_block, c_review, c_block
            )
            rows.append({
                "t_review": t_review, "t_block": t_block, "total_cost": cost,
                "n_review": n_rev, "n_block": n_blk, "fraud_missed": n_missed,
            })
    return pd.DataFrame(rows).sort_values("total_cost").reset_index(drop=True)

tier3_df = sweep_3tier(C_REVIEW, C_BLOCK)
best_3tier = tier3_df.iloc[0]

print("\n=== 3-Aksiyonlu model (Ca=$5, Cblock=$25) — ilk 10 sonuç ===")
print(tier3_df.head(10).to_string(index=False))

print(f"\nEn iyi kombinasyon: t_review={best_3tier['t_review']:.2f}, t_block={best_3tier['t_block']:.2f}")
print(f"Toplam maliyet: ${best_3tier['total_cost']:,.2f}")
print(f"REVIEW'a giden: {int(best_3tier['n_review'])}, BLOCK edilen: {int(best_3tier['n_block'])}, kaçırılan fraud: {int(best_3tier['fraud_missed'])}")

In [ ]:
# --- Adım 5: Eşdeğerlik doğrulaması ---
equiv_df = sweep_3tier(C_ADMIN_BAHNSEN, C_ADMIN_BAHNSEN)
equiv_diag = equiv_df[np.isclose(equiv_df["t_review"], equiv_df["t_block"])].sort_values("t_review")

check = pd.merge(
    bahnsen_df[["threshold", "total_cost"]],
    equiv_diag[["t_review", "total_cost"]].rename(columns={"t_review": "threshold", "total_cost": "total_cost_3tier"}),
    on="threshold",
)
max_diff = (check["total_cost"] - check["total_cost_3tier"]).abs().max()
print(f"Eşdeğerlik kontrolü — max fark: {max_diff:.10f} (0 olmalı)")
print("Doğrulandı: t_review=t_block VE c_review=c_block olduğunda 3-aksiyonlu model = Bahnsen 2-sınıflı model.\n")

# --- Adım 6: Duyarlılık analizi (Ca, Cblock senaryoları) ---
scenarios = {
    "Düşük":  (3.0, 15.0),
    "Base":   (5.0, 25.0),
    "Yüksek": (10.0, 50.0),
}

sensitivity_rows = []
for name, (c_rev, c_blk) in scenarios.items():
    df_s = sweep_3tier(c_rev, c_blk)
    best = df_s.iloc[0]
    sensitivity_rows.append({
        "senaryo": name, "C_review": c_rev, "C_block": c_blk,
        "en_iyi_t_review": best["t_review"], "en_iyi_t_block": best["t_block"],
        "toplam_maliyet": best["total_cost"],
        "n_review": int(best["n_review"]), "n_block": int(best["n_block"]),
        "kacirilan_fraud": int(best["fraud_missed"]),
    })

sensitivity_df = pd.DataFrame(sensitivity_rows)
print("=== Duyarlılık Analizi ===")
print(sensitivity_df.to_string(index=False))

# MLflow'a logla
with mlflow.start_run(run_name="ThresholdOptimization"):
    mlflow.log_param("model", "LightGBM-Tuned (Fold5 retrain)")
    mlflow.log_param("bahnsen_C_admin", C_ADMIN_BAHNSEN)
    mlflow.log_metric("bahnsen_best_threshold", float(best_bahnsen["threshold"]))
    mlflow.log_metric("bahnsen_best_cost", float(best_bahnsen["total_cost"]))
    mlflow.log_param("tier3_C_review", C_REVIEW)
    mlflow.log_param("tier3_C_block", C_BLOCK)
    mlflow.log_metric("tier3_best_t_review", float(best_3tier["t_review"]))
    mlflow.log_metric("tier3_best_t_block", float(best_3tier["t_block"]))
    mlflow.log_metric("tier3_best_cost", float(best_3tier["total_cost"]))
    for _, row in sensitivity_df.iterrows():
        mlflow.log_metric(f"sensitivity_{row['senaryo']}_cost", row["toplam_maliyet"])

print("\nMLflow'a loglandı (run: ThresholdOptimization).")

### Bulgu: BLOCK, REVIEW Tarafından Domine Ediliyor

Duyarlılık analizinde üç senaryonun da `t_block=0.99` (aramadaki en yüksek aday değer) bulması tesadüf değil, **matematiksel bir sonuç**:

Maliyet modelimizde REVIEW ve BLOCK'a **aynı fraud-durdurma etkinliği** (%100 — ikisi de fraud'u yakalıyor kabul edildi) atanmışken, REVIEW'un maliyeti (`Ca`) her zaman BLOCK'unkinden (`Cblock`) düşük. Yani herhangi bir işlem için:

`Cost(REVIEW) < Cost(BLOCK)` — HER durumda, koşulsuz.

Bu yüzden BLOCK, tanımlanan maliyet fonksiyonunda REVIEW tarafından **domine ediliyor** — optimizasyon BLOCK'u mümkün olduğunca az kullanmak istiyor, arama sınırı 0.99'da bittiği için oraya dayanıyor (gerçek matematiksel optimum muhtemelen t_block→1.0, yani "asla blokla").

**Sonuç ve sınırlılık:** Mevcut maliyet varsayımları altında 3-aksiyonlu sistem ekonomik olarak gerçek anlamda 3-aksiyonlu değil — BLOCK'un anlamlı bir rol oynaması için REVIEW'un kapasite kısıtı, %100'den düşük bir yakalama oranı, ya da aksiyon maliyetlerinin başka türlü farklılaştırılması gibi ek operasyonel varsayımlar gerekir. Bunu v1 kapsamında bilinçli olarak eklemedik (yeni bir keyfi varsayım eklemek yerine dürüst bir sınırlılık olarak belgelemeyi tercih ettik) — gelecekte `review_detection_rate` gibi bir parametreyle genişletilebilir.

## SHAP — Açıklanabilirlik

**Kavram:** SHAP, her özelliğin modelin tahminine (log-odds skalasında) ne kadar katkı sağladığını oyun teorisi (Shapley değerleri) ile adil şekilde paylaştırıyor. Ağaç modelleri için `TreeExplainer` çok hızlı çalışıyor. Threshold optimizasyonunda eğittiğimiz `threshold_model`'ı (Fold 5 train'inde, tuned hiperparametrelerle) açıklıyoruz — `X_val_f5` üzerinde, modelin görmediği veri.

**İki analiz:** (1) Global — model genelinde hangi özellikler en etkili, bizim 9 özelliğimiz anonim sütunlarla nasıl kıyaslanıyor; (2) Local — tek bir işlem için "neden bu karar verildi" açıklaması.

In [ ]:
import shap

explainer = shap.TreeExplainer(threshold_model)

# Global SHAP için örneklem (hız için tüm 83K satırı değil, 3000 rastgele satır)
np.random.seed(42)
sample_idx = np.random.choice(len(X_val_f5), size=min(3000, len(X_val_f5)), replace=False)
X_shap_sample = X_val_f5.iloc[sample_idx].reset_index(drop=True)
y_sample = y_val_f5[sample_idx]

# Not: Bu shap sürümünde LightGBM binary classifier için shap_values tek bir
# 2B array olarak dönüyor (pozitif sınıfa göre log-odds katkısı).
shap_values_pos = explainer.shap_values(X_shap_sample)
print("shap_values_pos shape:", shap_values_pos.shape)
print("expected_value (base log-odds):", explainer.expected_value)

In [ ]:
# --- Global SHAP önem sıralaması ---
mean_abs_shap = np.abs(shap_values_pos).mean(axis=0)
shap_importance = pd.Series(mean_abs_shap, index=X_shap_sample.columns).sort_values(ascending=False)

def categorize(feat):
    if feat in engineered_features:
        return "ENGINEERED"
    elif feat in interpretable_raw:
        return "INTERPRETABLE"
    else:
        return "ANONIM"

print("=== Global SHAP Önem Sıralaması (ilk 25) ===")
for i, (feat, val) in enumerate(shap_importance.head(25).items(), 1):
    print(f"{i:>2}. {feat:<30} {val:.4f}  [{categorize(feat)}]")

cat_totals = shap_importance.groupby(shap_importance.index.map(categorize)).sum()
cat_counts = pd.Series(
    [len(engineered_features), len(interpretable_raw), len(final_features) - len(engineered_features) - len(interpretable_raw)],
    index=["ENGINEERED", "INTERPRETABLE", "ANONIM"],
)
print("\n=== Kategori bazında toplam |SHAP| katkısı ===")
for cat in ["ENGINEERED", "INTERPRETABLE", "ANONIM"]:
    print(f"{cat:<15} toplam={cat_totals.get(cat, 0):.2f}  (n_features={cat_counts[cat]}, ortalama={cat_totals.get(cat, 0)/cat_counts[cat]:.4f})")

plt.figure()
shap.summary_plot(shap_values_pos, X_shap_sample, show=False, max_display=20)
plt.tight_layout()
plt.savefig("../reports/shap_summary.png", dpi=100, bbox_inches="tight")
plt.close()
print("\nGrafik kaydedildi: ../reports/shap_summary.png")

In [ ]:
# --- Türettiğimiz 9 özelliğin tam sırası (120 içinde) ---
full_rank = shap_importance.rank(ascending=False).astype(int)
print("=== Tüm ENGINEERED özelliklerin sırası (120 içinde) ===")
for feat in engineered_features:
    print(f"{full_rank[feat]:>3}. {feat:<30} SHAP={shap_importance[feat]:.4f}")

# --- Local SHAP: gerçek bir fraud işlemi ---
proba_sample = threshold_model.predict_proba(X_shap_sample)[:, 1]

tp_candidates = np.where((y_sample == 1) & (proba_sample > 0.5))[0]
example_idx = int(tp_candidates[np.argmax(proba_sample[tp_candidates])])

print(f"\n=== Örnek işlem (doğru yakalanan fraud) ===")
print(f"Gerçek etiket: fraud, tahmin edilen olasılık: {proba_sample[example_idx]:.4f}")

example_shap = pd.Series(shap_values_pos[example_idx], index=X_shap_sample.columns).sort_values(key=abs, ascending=False)
print("\nBu tahmine en çok katkı yapan ilk 10 özellik:")
for feat, val in example_shap.head(10).items():
    actual_val = X_shap_sample.iloc[example_idx][feat]
    yon = "RİSKİ ARTIRDI" if val > 0 else "RİSKİ AZALTTI"
    print(f"  {feat:<30} değer={actual_val}  SHAP={val:+.4f}  ({yon})")

plt.figure()
shap.waterfall_plot(shap.Explanation(
    values=shap_values_pos[example_idx],
    base_values=explainer.expected_value,
    data=X_shap_sample.iloc[example_idx],
    feature_names=X_shap_sample.columns.tolist(),
), show=False, max_display=15)
plt.tight_layout()
plt.savefig("../reports/shap_waterfall_example.png", dpi=100, bbox_inches="tight")
plt.close()
print("\nGrafik kaydedildi: ../reports/shap_waterfall_example.png")

### SHAP Bulguları — Feature Engineering'in İkinci (ve Daha Nüanslı) Sınavı

Ablation çalışmasında ("Feature Set Deneyleri") 9 özelliğimizin PR-AUC'a ölçülebilir bir katkı eklemediğini bulmuştuk. SHAP, aynı soruya farklı bir açıdan (agregat metrik değil, modelin HER özelliği ne kadar kullandığı) bakıyor ve daha nüanslı bir tablo ortaya koyuyor:

- **`merchant_risk`: 120 sütun arasında 11. sırada** — modelin gerçekten güvendiği, güçlü bir sinyal. Hem yüksek SHAP değeri hem de (V204, C13 gibi anonim kodların aksine) tam anlamıyla açıklanabilir olması onu SHAP raporunun en değerli özelliklerinden biri yapıyor.
- **`avg_transaction_amount`, `amount_deviation_from_user`, `dist1_deviation_from_user`, `time_since_last_transaction`**: orta sırada (36-44), makul ama baskın olmayan bir katkı.
- **`transactions_last_10min`, `new_device`, `new_location`, `transactions_last_24h`**: 114-119. sıralarda (120 üzerinden neredeyse en alt) — model bunları büyük ölçüde görmezden geliyor, muhtemelen anonim `C`/`V` sütunları aynı velocity/davranış sinyalini zaten daha iyi kodluyor.

**Sonuç:** Feature engineering'in başarısı tek boyutlu değil — `merchant_risk` net bir kazanım, diğer bazı özellikler (özellikle velocity/yenilik sinyalleri) modelin zaten sahip olduğu bilgiyle örtüştüğü için düşük katkı sağlıyor. Bu, raporda "hangi hipotezimiz doğrulandı, hangisi doğrulanmadı" şeklinde dürüstçe anlatılabilecek olgun bir bulgu.

**Local örnek:** Olasılığı 0.9994 olan (neredeyse kesin) bir fraud işleminde, riski en çok artıran etkenler çoğunlukla anonim `V`/`C` sütunları (`V258`, `V45`, `C1`, `C11`) ama **`merchant_risk` de ilk 5 sırada** (+0.77 katkı) — yani tekil örnek düzeyinde de bizim özelliğimiz karar sürecinin görünür bir parçası.

## Nihai Holdout Test Değerlendirmesi

Tüm kararlar (model, hiperparametreler, threshold'lar) validation üzerinde kilitlendi — `df_holdout_test`'e (en yeni %15, hiçbir aşamada dokunulmadı) **şimdi TEK SEFER** bakıyoruz. Bu, projenin gerçek/tarafsız son notu.

Nihai model: tuned LightGBM (`num_leaves=63, n_estimators=300, learning_rate=0.1, min_child_samples=10`), tüm `df_cv_pool`'a fit edilmiş hali (MLflow'dan `runs:/96e87015b55547659886a6037f8ab5dd/model` ile geri yüklendi — kurtarma reçetesinin gerçek kullanımı). Threshold: Base senaryo (`t_review=0.23, t_block=0.99, C_review=$5, C_block=$25`).

In [ ]:
import mlflow.lightgbm

# Tuned LightGBM'i MLflow'dan geri yükle (tüm df_cv_pool'a fit edilmiş hali)
final_model = mlflow.lightgbm.load_model("runs:/96e87015b55547659886a6037f8ab5dd/model")

# df_holdout_test'e AYNI ön-işleme adımlarını uygula (df_cv_pool'a uyguladığımız gibi)
obj_cols_h = df_holdout_test.select_dtypes(include="object").columns
df_holdout_test[obj_cols_h] = df_holdout_test[obj_cols_h].astype("category")

bool_cols_h = df_holdout_test.select_dtypes(include="boolean").columns
df_holdout_test[bool_cols_h] = df_holdout_test[bool_cols_h].astype("float")

df_holdout_test["dist1_deviation_from_user"] = df_holdout_test["dist1_deviation_from_user"].replace(
    [np.inf, -np.inf], np.nan
)

X_holdout = df_holdout_test[final_features]
y_holdout = df_holdout_test["isFraud"].values
amt_holdout = df_holdout_test["TransactionAmt"].values

y_proba_holdout = final_model.predict_proba(X_holdout)[:, 1]

pr_auc_holdout = average_precision_score(y_holdout, y_proba_holdout)
print(f"=== NİHAİ HOLDOUT PR-AUC: {pr_auc_holdout:.4f} ===")
print(f"(Karşılaştırma için: walk-forward CV ortalaması 0.5681'di — aşırı öğrenme yok, model iyi genelliyor)")

In [ ]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score

# Base senaryonun kilitlenen eşikleri (validation'da bulunmuştu)
T_REVIEW_FINAL = 0.23
T_BLOCK_FINAL = 0.99
C_REVIEW_FINAL = 5.0
C_BLOCK_FINAL = 25.0

is_approve = y_proba_holdout < T_REVIEW_FINAL
is_review = (y_proba_holdout >= T_REVIEW_FINAL) & (y_proba_holdout < T_BLOCK_FINAL)
is_block = y_proba_holdout >= T_BLOCK_FINAL

print("=== Holdout — Aksiyon Dağılımı ===")
print(f"APPROVE: {is_approve.sum():>6,} ({is_approve.mean()*100:.1f}%)")
print(f"REVIEW:  {is_review.sum():>6,} ({is_review.mean()*100:.1f}%)")
print(f"BLOCK:   {is_block.sum():>6,} ({is_block.mean()*100:.1f}%)")

fraud_in_approve = (y_holdout[is_approve] == 1).sum()
fraud_in_review = (y_holdout[is_review] == 1).sum()
fraud_in_block = (y_holdout[is_block] == 1).sum()
total_fraud = y_holdout.sum()

print(f"\nToplam fraud (holdout): {total_fraud}")
print(f"APPROVE'da kaçan fraud: {fraud_in_approve} ({fraud_in_approve/total_fraud*100:.1f}%)")
print(f"REVIEW'da yakalanan fraud: {fraud_in_review} ({fraud_in_review/total_fraud*100:.1f}%)")
print(f"BLOCK'ta yakalanan fraud: {fraud_in_block} ({fraud_in_block/total_fraud*100:.1f}%)")

y_pred_flagged = (is_review | is_block).astype(int)
print("\n=== Standart Sınıflandırma Metrikleri (flagged vs approve) ===")
print(f"Precision: {precision_score(y_holdout, y_pred_flagged):.4f}")
print(f"Recall:    {recall_score(y_holdout, y_pred_flagged):.4f}")
print(f"F1:        {f1_score(y_holdout, y_pred_flagged):.4f}")
print("\nConfusion Matrix (satır=gerçek, sütun=tahmin [0=approve,1=flagged]):")
print(confusion_matrix(y_holdout, y_pred_flagged))

final_cost, n_rev, n_blk, n_missed = compute_total_cost(
    y_holdout, amt_holdout, y_proba_holdout,
    T_REVIEW_FINAL, T_BLOCK_FINAL, C_REVIEW_FINAL, C_BLOCK_FINAL,
)
print(f"\n=== NİHAİ MALİYET (holdout, gerçek TransactionAmt ile) ===")
print(f"Toplam maliyet: ${final_cost:,.2f}")
print(f"İşlem başına ortalama maliyet: ${final_cost/len(y_holdout):.4f}")

naive_cost = amt_holdout[y_holdout == 1].sum()
print(f"\nKarşılaştırma — model YOKSA (hepsi approve, tüm fraud kayıp): ${naive_cost:,.2f}")
print(f"Modelin sağladığı tasarruf: ${naive_cost - final_cost:,.2f} ({(1 - final_cost/naive_cost)*100:.1f}%)")

with mlflow.start_run(run_name="FinalHoldoutEvaluation"):
    mlflow.log_param("model", "LightGBM-Tuned (full df_cv_pool fit)")
    mlflow.log_param("t_review", T_REVIEW_FINAL)
    mlflow.log_param("t_block", T_BLOCK_FINAL)
    mlflow.log_param("C_review", C_REVIEW_FINAL)
    mlflow.log_param("C_block", C_BLOCK_FINAL)
    mlflow.log_metric("holdout_pr_auc", pr_auc_holdout)
    mlflow.log_metric("holdout_precision", precision_score(y_holdout, y_pred_flagged))
    mlflow.log_metric("holdout_recall", recall_score(y_holdout, y_pred_flagged))
    mlflow.log_metric("holdout_f1", f1_score(y_holdout, y_pred_flagged))
    mlflow.log_metric("holdout_total_cost", final_cost)
    mlflow.log_metric("holdout_naive_cost", naive_cost)
    mlflow.log_metric("holdout_savings_pct", (1 - final_cost/naive_cost)*100)
    mlflow.log_metric("holdout_fraud_missed_pct", fraud_in_approve/total_fraud*100)

print("\nMLflow'a loglandı (run: FinalHoldoutEvaluation).")

## Sonuç

**Nihai model:** LightGBM (tuned: `num_leaves=63, n_estimators=300, learning_rate=0.1, min_child_samples=10`), Top-100 feature seti (9 engineered + 11 interpretable + 100 top anonim).

**Holdout test sonuçları (hiç dokunulmamış %15'lik dilim, tek seferlik):**
- PR-AUC: **0.5500** (walk-forward CV ortalaması 0.5681'e çok yakın — aşırı öğrenme yok, sağlam genelleme)
- Recall: %83.5 (fraud'un çoğunu yakalıyor), Precision: %16.1 (dengesiz veri setinde beklenen düşüklük)
- **Toplam maliyet: $174,725** vs model olmasaydı **$469,609** — **%62.8 tasarruf**

**Proje boyunca öne çıkan üç dürüst/olgun bulgu:**
1. 9 türettiğimiz özellik, ham+anonim sütunların zaten taşıdığı sinyale ölçülebilir bir PR-AUC katkısı eklemedi (ablation) — ama SHAP'ta `merchant_risk` 120 sütun arasında 11. sırada çıkıp gerçekten kullanıldığını gösterdi; bazı özellikler (yenilik/hız sinyalleri) ise modelin zaten sahip olduğu bilgiyle örtüştü.
2. Random Forest'ta hyperparameter tuning işe yaramadı (hatta kötüleşti), LightGBM'de işe yaradı — "tuning her zaman iyileştirir" varsayımının yanlış olduğunu gösteren somut bir örnek.
3. 3-aksiyonlu (APPROVE/REVIEW/BLOCK) maliyet modelinde BLOCK, REVIEW tarafından domine edildi — mevcut varsayımlar altında sistem ekonomik olarak gerçek anlamda 3-aksiyonlu değil; bu bilinçli bir v1 sınırlılığı olarak belgelendi.

Modelleme aşaması tamamlandı. Sıradaki adımlar: FastAPI ML servisi, Spring Boot backend entegrasyonu, Docker Compose, frontend.

## Model Registry ve Export

İki şey yapıyoruz: (1) MLflow Model Registry'de bu modeli resmi bir "production" versiyonu olarak işaretlemek (versiyon takibi/izlenebilirlik için), (2) modeli ve serving için gereken tüm metadata'yı (feature listesi, dtype bilgisi, threshold'lar, holdout metrikleri) `ml/models/` altına export etmek — gelecekteki FastAPI ML servisi bunu doğrudan yükleyip kullanacak. `ml/models/` (tıpkı `ml/data/` gibi) `.gitignore`'da — büyük binary model dosyaları git'e değil, MLflow registry'ye/artifact store'a emanet.

In [ ]:
client = mlflow.tracking.MlflowClient()

MODEL_NAME = "fraud-detection-lightgbm"
RUN_ID = "96e87015b55547659886a6037f8ab5dd"
model_uri = f"runs:/{RUN_ID}/model"

try:
    client.create_registered_model(MODEL_NAME)
except mlflow.exceptions.MlflowException:
    pass  # zaten var

mv = client.create_model_version(name=MODEL_NAME, source=model_uri, run_id=RUN_ID)
client.set_registered_model_alias(MODEL_NAME, "production", mv.version)
client.update_model_version(
    name=MODEL_NAME,
    version=mv.version,
    description=(
        "Tuned LightGBM (num_leaves=63, n_estimators=300, learning_rate=0.1, "
        "min_child_samples=10), Top-100 feature seti. Holdout PR-AUC=0.5500. "
        "Threshold: t_review=0.23, t_block=0.99 (Base senaryo, Ca=$5, Cblock=$25)."
    ),
)
print(f"'{MODEL_NAME}' — versiyon {mv.version} — 'production' alias'i atandı.")

In [ ]:
import joblib
import json
import os
from datetime import datetime, timezone

os.makedirs("../models", exist_ok=True)

joblib.dump(final_model, "../models/fraud_lightgbm_v1.joblib")

categorical_cols_final = list(df_cv_pool[final_features].select_dtypes(include="category").columns)
numeric_cols_final = [c for c in final_features if c not in categorical_cols_final]

# ÖNEMLİ: LightGBM, kategorik sütunları pandas'ın ürettiği tamsayı KODLARI
# üzerinden işliyor (etiket değil). Serving tarafında tek satırlık bir
# DataFrame'i .astype("category") yaparsan, pandas o satırda sadece görülen
# tek değere göre YENİ kodlar üretir — bunlar eğitimdeki kodlarla eşleşmez ve
# SESSİZCE yanlış tahminlere yol açar. Bu yüzden eğitim sırasındaki tam
# kategori sırasını (df_cv_pool[col].cat.categories) da kaydediyoruz; serving
# tarafı pd.Categorical(deger, categories=bu_liste) ile AYNI kodlamayı
# yeniden üretecek.
category_mappings = {c: list(df_cv_pool[c].cat.categories) for c in categorical_cols_final}

metadata = {
    "model_name": MODEL_NAME,
    "model_version": mv.version,
    "mlflow_run_id": RUN_ID,
    "trained_at": datetime.now(timezone.utc).isoformat(),
    "hyperparameters": {
        "num_leaves": 63, "n_estimators": 300,
        "learning_rate": 0.1, "min_child_samples": 10,
        "class_weight": "balanced",
    },
    "final_features": final_features,
    "numeric_columns": numeric_cols_final,
    "categorical_columns": categorical_cols_final,
    "category_mappings": category_mappings,
    "thresholds": {
        "t_review": 0.23, "t_block": 0.99,
        "c_review": 5.0, "c_block": 25.0,
        "scenario": "Base",
    },
    "holdout_metrics": {
        "pr_auc": 0.5500,
        "precision": float(precision_score(y_holdout, y_pred_flagged)),
        "recall": float(recall_score(y_holdout, y_pred_flagged)),
        "f1": float(f1_score(y_holdout, y_pred_flagged)),
    },
    "notes": (
        "merchant_risk feature'i canlı sistemde eğitim setinden üretilip "
        "dondurulmuş (frozen) bir lookup tablosu olarak servis edilmelidir, "
        "her istekte yeniden hesaplanamaz (bkz. features.py docstring)."
    ),
}

with open("../models/fraud_lightgbm_v1_metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Kaydedildi:")
print(" -", os.path.abspath("../models/fraud_lightgbm_v1.joblib"))
print(" -", os.path.abspath("../models/fraud_lightgbm_v1_metadata.json"))